# Wolf-of-Wallstreet — train the model on Google Colab

**3 steps:**
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Put your **`trading-agent`** folder in Google Drive (My Drive). Keep `scripts/`, `backend/`, `requirements.txt`, **and your `.env`** (it holds your Alpaca keys + the tuned training knobs). Delete `frontend/`, `backend/.venv/`, `.git/`, `node_modules/` to keep the upload small.
3. **Runtime → Run all.** (A Drive auth popup appears on the first cell — approve it.)

The trained weights are written to **`models/trading_lstm_latest.pt`** inside your Drive folder, so they survive disconnects. Bring that file back to your PC's `trading-agent/models/`.

> **RAM (Colab ≈ 12.7 GB):** the train cell uses `--mmap`, which streams a float16 dataset from local disk instead of holding it in RAM — so **all 13 symbols × multiple years fit**. The limits become Colab's disk (~107 GB) and your 12 h / 30 GPU-hr budget, not RAM.
>
> **Consistency:** the model core + regularization come from your uploaded `.env`, so training and live trading use identical settings (a checkpoint trained one way won't load the other). Don't override them here.

### 1) Get the project into the runtime (Google Drive)
Edit `PROJECT_DIR` if your folder is named/placed differently. *(Prefer GitHub? Replace this cell with `!git clone https://github.com/<you>/<repo>.git` and set `PROJECT_DIR` to the cloned `.../trading-agent`.)*

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/trading-agent"  # <-- edit if needed

from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.exists(os.path.join(PROJECT_DIR, 'scripts', 'pretrain.py')), \
    f"Couldn't find scripts/pretrain.py under {PROJECT_DIR}. Check the folder name/location in your Drive."
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())
print('.env present:', os.path.exists('.env'), '(needed for stocks + tuned knobs)')
!ls

### 2) Install the (few) training dependencies
Colab already ships the GPU build of PyTorch + pandas, so we only add the small extras. `numpy<2` keeps `pandas_ta` working; the training runs in a fresh `python` process that picks up this pinned numpy automatically.

In [ ]:
!pip install -q "numpy<2" pandas_ta structlog python-dotenv pydantic pydantic-settings requests

### 3) Confirm the GPU is on

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU! Set Runtime → Change runtime type → T4 GPU, then Run all again.')

### 4) Training knobs (read from your uploaded `.env`)
These are already tuned for generalization + risk-adjusted returns. The trainer reads them from `.env`, so they match what you'll run live. This cell only shows them — **don't override here** or the checkpoint won't match the live model.

In [ ]:
from dotenv import dotenv_values
cfg = dotenv_values('.env')
for k in ['NN_RNN_TYPE','NN_DROPOUT','NN_WEIGHT_DECAY','NN_LABEL_SMOOTHING','NN_AUGMENT_NOISE_STD']:
    print(f'{k:22s} = {cfg.get(k)}')
print('\nReward/sizing (live):', {k: cfg.get(k) for k in ['NN_KELLY_FRACTION','NN_MIN_EDGE_OVER_FEE','RISK_MAX_POSITION_PCT']})
assert cfg.get('ALPACA_API_KEY'), 'No ALPACA_API_KEY in .env — the 5 stocks will be skipped.'

### 5) Train ALL 13 symbols (8 crypto + 5 stocks) in one pass
`--mmap` streams a float16 dataset from local disk, so RAM is no longer the limit — you can train every symbol over multiple years on Colab's 12.7 GB. Earlier `START_YEAR` = more data = longer run (watch the 12 h / 30 GPU-hr budget).

In [ ]:
START_YEAR = 2022          # mmap removes the RAM cap — go back as far as your time budget allows
EPOCHS     = 30
PATIENCE   = 5             # early stopping
BATCH_SIZE = 256
# All 13: 8 crypto + 5 stocks. Stocks load your Alpaca keys from the uploaded .env.
SYMBOLS    = "BTCUSDT ETHUSDT SOLUSDT AAVEUSDT XLMUSDT XRPUSDT ADAUSDT DOGEUSDT SNDK AMD MU AXTI BE"

!python scripts/pretrain.py --mmap --mmap-dir /content/_mmap_cache \
    --start-year {START_YEAR} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch-size {BATCH_SIZE} --symbols {SYMBOLS}

### 6) Verify the checkpoint
It's at `models/trading_lstm_latest.pt` inside your Drive folder — download it from Drive and drop it into your PC's `trading-agent/models/`.

In [ ]:
import os
ckpt = 'models/trading_lstm_latest.pt'
if os.path.exists(ckpt):
    print(f'✓ {ckpt}  ({os.path.getsize(ckpt)/1e6:.1f} MB)  — saved in your Drive.')
else:
    print('No checkpoint yet — did training finish without error above?')